# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Jesper Flunkert||| |
| Member | Nina Alexandra Klünsch| |
| Member *(optional)* | | |


## 2. Mission Title & Research Question

**Title:** *Predicting Future Profits: A Machine Learning Approach to R&D Spending*

**Research question:**  
*Does current R&D spending causally drive a firm’s profitability three years later (t+3), and can machine learning predict these high growth outliers based on their primary investment strategy (Build, Buy, or Equip)?*

**Why it matters:**  
*Evaluating R&D efficiency presents a clear analytical problem: it reduces earnings today, while the financial returns only appear years later. Furthermore, it is difficult to separate cause and effect. Does R&D actually create future prfits, or do already dominant companies simply have more excess cash to spend on research? By using machine learning to model these delayed effects, we can objectively separate true innovators from companies that simply waste capital.
In addition we can leverage our results in order to alayse the usefulness of different strategys in R&D.*


## 3. Data

**Source(s):** * • WRDS - Compustat North America (Fundamentals Annual): Timeframe 01.2010–05.2026

**Unit of observation:** *Firm-Year*

**Key variables (Raw Data Output):**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
| :--- | :--- | :--- | :--- |
| **gvkey** | Identifier | Index | Global Company Key |
| **conm** | Text | Descriptive | Company Name |
| **tic** | Text | Descriptive | Ticker Symbol |
| **sic** | Categorical | Filter | Standard Industrial Classification Code |
| **fyear** | Integer | Time Index | Data Year |
| **at** | Continuous | Feature | Total Assets |
| **oibdp** | Continuous | Feature | Operating Income Before Depreciation |
| **revt** | Continuous | Feature | Total Revenue |
| **xrd** | Continuous | Feature | Research and Development Expense |
| **aqc** | Continuous | Feature | Acquisitions |
| **capx** | Continuous | Feature | Capital Expenditures |

**Potential data quality issues:** 
* **Missing Values in R&D (xrd):** SOme companies leave the R&D field blank if they have no expenses, resulting in `NaN` rather than `0`. We must  impute `NaN` as `0`.
* **Survivorship:** If a company burns its capital on R&D and goes bankrupt before year $t+3$, we need to capture that failure. If we only analyze active companies, R&D will look much more successful than it actually is. That’s why we make sure to include inactive or delisted firms in our dataset.
* **Excluding the Financial Sector (Selection Bias):** Financial institutions operate completely different than industrial firms. Because they hold massive assets but practically spend no money on R&D, they act as massive outliers that would ruin our intensity metrics. We will solve this by dropping the financial sector during the initial data cleaning.

In [ ]:
import pandas as pd
import numpy as np

file_path = r"..." 

df = pd.read_csv(file_path, low_memory=False)

print(f"shape: {df.shape}")

print(f"head: {df.head()}")

print(f"info: {df.info()}")

print(f"describe: {df.describe()}")

"""
Return: 
    shape: (155740, 16)
    head:   costat curcd datafmt indfmt consol     tic  fyear  gvkey  \
    0      A   USD     STD   INDL      C     AIR   2010   1004   
    1      I   USD     STD   INDL      C  ADCT.1   2010   1013   
    2      I   USD     STD   INDL      C    AFAP   2010   1019   
    3      A   USD     STD   INDL      C     AAL   2010   1045   
    4      A   USD     STD   INDL      C    CECO   2010   1050   

                            conm   sic         at     oibdp       revt     xrd  \
    0                     AAR CORP  5080   1703.727   196.312   1775.782     NaN   
    1   ADC TELECOMMUNICATIONS INC  3661   1474.500   121.400   1156.600  69.700   
    2   AFA PROTECTIVE SYSTEMS INC  7380     29.546     4.042     72.433     NaN   
    3  AMERICAN AIRLINES GROUP INC  4512  25088.000  1303.000  22170.000     NaN # <- as mentioned above    
    4      CECO ENVIRONMENTAL CORP  3564     74.791     6.398    140.602   0.053   

    aqc      capx  
    0  0.0   124.879  
    1  4.4    29.400  
    2  0.0     1.103  
    3  0.0  1962.000  
    4  0.0     0.654  
    <class 'pandas.core.frame.DataFrame'>
    RangeIndex: 155740 entries, 0 to 155739
    Data columns (total 16 columns):
    #   Column   Non-Null Count   Dtype  
    ...
    25%         0.232000  
    50%         5.524000  
    75%        70.063000  
    max    131819.000000  
"""

shape: (155740, 16)
head:   costat curcd datafmt indfmt consol     tic  fyear  gvkey  \
0      A   USD     STD   INDL      C     AIR   2010   1004   
1      I   USD     STD   INDL      C  ADCT.1   2010   1013   
2      I   USD     STD   INDL      C    AFAP   2010   1019   
3      A   USD     STD   INDL      C     AAL   2010   1045   
4      A   USD     STD   INDL      C    CECO   2010   1050   

                          conm   sic         at     oibdp       revt     xrd  \
0                     AAR CORP  5080   1703.727   196.312   1775.782     NaN   
1   ADC TELECOMMUNICATIONS INC  3661   1474.500   121.400   1156.600  69.700   
2   AFA PROTECTIVE SYSTEMS INC  7380     29.546     4.042     72.433     NaN   
3  AMERICAN AIRLINES GROUP INC  4512  25088.000  1303.000  22170.000     NaN   
4      CECO ENVIRONMENTAL CORP  3564     74.791     6.398    140.602   0.053   

   aqc      capx  
0  0.0   124.879  
1  4.4    29.400  
2  0.0     1.103  
3  0.0  1962.000  
4  0.0     0.654  
<class

## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [X] Causal graph / DAG (DoWhy)
- [ ] Backdoor adjustment
- [ ] Instrumental variable
- [X] Propensity score stratification
- [ ] Other: ___

*Justification: Highly dominant firms naturally spend more on R&D, making it difficult to isolate true cause and effect using basic regression. We will construct a DAG where Firm_Size, Current_Profitability, and Industry_Code act as backdoor paths. We will define the Treatment as "High R&D Intensity" (e.g. top quartile) and use Propensity Score Stratification to isolate the Average Treatment Effect on the Fwd_Profitability (t+3 profit margin).*

### 4b. Supervised Learning
- [ ] Linear / Ridge / Lasso regression
- [X] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [X] Decision Tree / Random Forest
- [ ] Neural network (regression or classification)
- [ ] Other: ___

*Justification: This research is framed as a classification task that predicts whether a firm will become a “High-Growth Winner,” defined as achieving a significant increase in profitability within the next three years (t+3). The goal is to examine whether R&D activity and innovation-related factors can help predict long-term profit growth. Random Forest is suitable because it can capture complex, non-linear relationships in financial data and handle interactions between variables more effectively than traditional linear models such as Linear, Ridge, Lasso, or Logistic Regression. 
Performance will be compared against a dummy classifier (majority-class prediction) and Logistic Regression as a traditional benchmark model. F1-Score and Recall will be used as evaluation measures.*

### 4c. Unsupervised Learning / Generative Models
- [X] K-Means clustering
- [X] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification: Firms allocate capital differently to drive growth. To categorize these strategies, we will apply Unsupervised Learning to our three core metrics at year t: RnD_Intensity (Build), Acquisition_Ratio (Buy), and CAPEX_Intensity (Equip). Hierarchical clustering will map the overarching structure of corporate strategies via a dendrogram, identifying distinct market types. K-Means will then classify all observations. The clustering quality will be assessed via Silhouette metrics. 
Crucially, this step bridges the methodology: the resulting cluster labels will serve as input features for both the Causal Inference DAG and the Supervised Gradient Boosting models.*


## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

* **Causal Inference (Validation & Sensitivity):** We will validate our DAG using DoWhys automated Refutation Test. Specifically, we will use a Placebo Treatment Refuter (replacing the R&D treatment with random noise) and an Unobserved Common Cause Refuter (to determine how strong an unmeasured confounder must be to invalidate our results).  

* **Supervised Learning (Metrics & Baselines):** Because our target variable ("High-Growth Winner") represents a minority of firms, standard accuracy would be misleading (imbalanced dataset). We will evaluate model performance using the F1-Score and Recall to effectively capture true innovators. Our Random Forest model will be benchmarked against two baselines:
  1. A Dummy Classifier (majority-class prediction to prove basic predictive validity).
  2. A Logistic Regression (serving as a traditional, linear econometric baseline to prove that non-linear ML models capture additional value).
* **Unsupervised Learning (Model Fit):** The structural validity of our distinct market types  will be evaluated mathematically using the Silhouette Score. We will also use the Elbow Method to quantitatively validate the optimal number of clusters (k).
(Additionally the clusters can be compared to a simple clustering by “high/low R&D spending, high/low growth, etc.” as a baseline)



## 6. Work Plan

preliminary:

| Step | Owner | Description |
|------|-------|-------------|
| 1 | Jesper | Data collection & cleaning |
| 2 | Nina | EDA |
| 3 | Jesper | Causal inference block |
| 4 | Nina | Supervised learning block |
| 5 | Jesper | Unsupervised / generative block |
| 6 | Nina | Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
